# Alternative Data

Download the observed AAPL news used by the alternative-data workflow, then inspect the normalized project schema. Existing Parquet data is reused so repeated notebook runs do not make unnecessary external requests.

## Download the Data

In [ ]:
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from src.data_preprocessing.alternative_data import save_alpaca_news

PROJECT_ROOT = Path.cwd().resolve().parents[1]
news_path = PROJECT_ROOT / "data/research_data/alternative/data/aapl_2025-01-01_2025-12-31.parquet"

if not news_path.is_file():
    save_alpaca_news(
        symbols=["AAPL"],
        start=datetime(2025, 1, 1, tzinfo=timezone.utc),
        end=datetime(2025, 12, 31, 23, 59, 59, 999999, tzinfo=timezone.utc),
        output_path=news_path,
    )

alternative_data = pd.read_parquet(news_path)
expected_columns = [
    "id", "headline", "source", "url", "summary",
    "created_at", "updated_at", "symbols", "author", "content",
]
assert alternative_data.columns.tolist() == expected_columns
alternative_data["created_at"] = pd.to_datetime(alternative_data["created_at"], utc=True)
alternative_data["updated_at"] = pd.to_datetime(alternative_data["updated_at"], utc=True)
assert alternative_data["symbols"].astype("string").str.contains("AAPL", na=False).all()
news_path

## Take a Quick Look at the Data Structure

In [ ]:
alternative_data.head()

In [ ]:
alternative_data.info()

In [ ]:
alternative_data["source"].value_counts(dropna=False)

In [ ]:
alternative_data[["id"]].describe()

In [ ]:
alternative_data[["id"]].hist(bins=50, figsize=(6, 4))
plt.tight_layout()
plt.show()